In [6]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

df = pd.read_csv("../Data/imf-dm-export-20260910.csv")
df = df.rename(columns={df.columns[0]: "country"})
uk = df[df["country"] == "United Kingdom"].iloc[0]

# year columns are strings in a CSV — match 4-digit names, not int type
years = [c for c in df.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
data = pd.DataFrame({"year": [int(y) for y in years],
                     "debt": [pd.to_numeric(uk[y], errors="coerce") for y in years]}).dropna()

def party(y):
    if y < 1997: return "Conservative"
    if y < 2010: return "Labour"
    if y < 2024: return "Conservative"
    return "Labour"
data["party"] = data["year"].apply(party)

spans, cur, start = [], data.iloc[0]["party"], data.iloc[0]["year"]
for _, r in data.iterrows():
    if r["party"] != cur:
        spans.append((cur, start, r["year"])); cur, start = r["party"], r["year"]
spans.append((cur, start, data["year"].max() + 1))
spans_df = pd.DataFrame(spans, columns=["party", "x0", "x1"])

data.tail()

,year,debt,party
47,2027,104.1,Labour
48,2028,103.9,Labour
49,2029,103.5,Labour
50,2030,102.9,Labour
51,2031,102.6,Labour


In [10]:
styles = EcoStyles(); styles.register_and_enable_theme()

CON, LAB, INK = "#0087DC", "#E4003B", "#122b39"
XDOM = [1980, data["year"].max() + 1]

# party bands
bg = alt.Chart(spans_df).mark_rect(opacity=0.14).encode(
    x=alt.X("x0:Q", scale=alt.Scale(domain=XDOM, nice=False),
            axis=alt.Axis(format="d", grid=False), title=None),
    x2="x1:Q",
    color=alt.Color("party:N",
        scale=alt.Scale(domain=["Conservative", "Labour"], range=[CON, LAB]),
        legend=alt.Legend(orient="top", direction="horizontal")))

# grey wash over the projection years (2025+)
proj = pd.DataFrame({"x0": [2025], "x1": [data["year"].max() + 1]})
proj_bg = alt.Chart(proj).mark_rect(opacity=0.10, color="#676A86").encode(x="x0:Q", x2="x1:Q")

# line: solid actual, dashed projection (overlap at 2025 so they join)
actual = data[data["year"] <= 2025]
projline = data[data["year"] >= 2025]
line_a = alt.Chart(actual).mark_line(color=INK, strokeWidth=2.2).encode(
    x="year:Q", y=alt.Y("debt:Q", title="Gross debt, % of GDP"))
line_p = alt.Chart(projline).mark_line(color=INK, strokeWidth=2.2, strokeDash=[3, 3]).encode(
    x="year:Q", y="debt:Q")

chart = (bg + proj_bg + line_a + line_p).properties(
    width=640, height=340,
    title=alt.Title("Debt and the party in power",
                    subtitle="UK gross debt, % of GDP · shading shows government · dashed = IMF projection"))

styles.save(chart, name="debt_by_party", svg=True)
chart

alt.LayerChart(...)